# Agent 1 — File Reception playground

Demonstrates `FileReceptionAgent` end-to-end:
validation → SHA-256 → MIME detection → SSE events → audit record.

> **Kernel**: select `.venv` (Python 3.12) in the top-right kernel picker.

## 1 — Imports

In [1]:
import asyncio

from classiflow.ingesta.agents import FileReceptionAgent
from classiflow.ingesta.mime import detect_mime
from classiflow.shared.audit.service import AuditService
from classiflow.shared.database.repositories.audit import InMemoryAuditRepository
from classiflow.shared.events.broadcaster import EventBroadcaster

print("imports OK")

imports OK


## 2 — Build the agent

`FileReceptionAgent` is a Pydantic model — pass dependencies as keyword arguments.
`detect_mime` is the production implementation (uses the `filetype` library).

In [2]:
audit_repo = InMemoryAuditRepository()
audit = AuditService(audit_repo)
broadcaster = EventBroadcaster()

agent = FileReceptionAgent(
    audit=audit,
    broadcaster=broadcaster,
    mime_detector=detect_mime,
)
print("agent ready")

agent ready


## 3 — Run with a valid PDF

The cell uses top-level `await` — that works in Jupyter without `asyncio.run()`.

In [3]:
MINIMAL_PDF = (
    b"%PDF-1.4\n1 0 obj\n<< /Type /Catalog >>\nendobj\n"
    b"xref\n0 1\n0000000000 65535 f\ntrailer\n<< /Size 1 >>\nstartxref\n9\n%%EOF"
)

result = await agent.run(job_id="demo-001", filename="sample.pdf", file_bytes=MINIMAL_PDF)

print("=== File state ===")
print(f"  passed          : {result.passed}")
print(f"  sha256          : {result.sha256}")
print(f"  detected_mime   : {result.detected_mime}")
print(f"  file_size_bytes : {result.file_size_bytes}")
print(f"  rejection_reason: {result.rejection_reason}")

2026-06-21 21:44:38.119 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=demo-001 agent=agent1_file_reception event=passed


=== File state ===
  passed          : True
  sha256          : 30fdc7230755e392c94368b7bc639c3824a5511c1d76622d31221bca53ffd11a
  detected_mime   : application/pdf
  file_size_bytes : 112
  rejection_reason: 


## 4 — Inspect the audit record

In [4]:
records = await audit_repo.list_for_job("demo-001")

print("=== Audit records ===")
for r in records:
    print(f"  event       : {r.event}")
    print(f"  agent       : {r.agent}")
    print(f"  duration_ms : {r.duration_ms} ms")
    print(f"  detail      : {r.detail}")
    print()

=== Audit records ===
  event       : passed
  agent       : agent1_file_reception
  duration_ms : 0 ms
  detail      : {'filename': 'sample.pdf', 'sha256': '30fdc7230755e392c94368b7bc639c3824a5511c1d76622d31221bca53ffd11a', 'detected_mime': 'application/pdf', 'file_size_bytes': 112, 'passed': True, 'rejection_reason': ''}



## 5 — Observe SSE events in real time

The agent emits `STARTED` then `PASSED`/`FAILED` through the `EventBroadcaster`.
Here we subscribe before calling `run()` so we catch both events.

In [5]:
broadcaster2 = EventBroadcaster()
agent2 = FileReceptionAgent(
    audit=AuditService(InMemoryAuditRepository()),
    broadcaster=broadcaster2,
    mime_detector=detect_mime,
)

events = []


async def collect() -> None:
    async for event in broadcaster2.subscribe("demo-002"):
        events.append(event)
        print(f"  SSE → agent={event.agent}  status={event.status}")


collect_task = asyncio.create_task(collect())
await asyncio.sleep(0)

await agent2.run(job_id="demo-002", filename="sample.pdf", file_bytes=MINIMAL_PDF)
await broadcaster2.close("demo-002")
await collect_task

print(f"\ncollected {len(events)} events")

2026-06-21 21:44:38.200 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=demo-002 agent=agent1_file_reception event=passed


  SSE → agent=agent1_file_reception  status=started
  SSE → agent=agent1_file_reception  status=passed

collected 2 events


## 6 — Rejection cases

Agent rejects: missing file, empty bytes, and anything over 50 MB.

In [6]:
cases = [
    ("no file", None),
    ("empty file", b""),
    ("oversized", b"x" * (51 * 1024 * 1024)),
]

print("=== Rejection cases ===")
for label, data in cases:
    r = await FileReceptionAgent(
        audit=AuditService(InMemoryAuditRepository()),
        broadcaster=EventBroadcaster(),
        mime_detector=detect_mime,
    ).run(job_id=f"demo-{label}", filename="test.pdf", file_bytes=data)
    print(f"  {label:12} → passed={r.passed}  reason='{r.rejection_reason}'")

2026-06-21 21:44:38.249 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=demo-no file agent=agent1_file_reception event=failed
2026-06-21 21:44:38.251 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=demo-empty file agent=agent1_file_reception event=failed
2026-06-21 21:44:38.251 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=demo-oversized agent=agent1_file_reception event=failed


=== Rejection cases ===
  no file      → passed=False  reason='No file provided'
  empty file   → passed=False  reason='File is empty'
  oversized    → passed=False  reason='File exceeds maximum allowed size of 50 MB'


## 7 — Run with a real file from disk

Drop any PDF, DOCX, or image into:
```
src/classiflow/playground/samples/
```
then set `FILE_NAME` below and run the cell.

In [8]:
from pathlib import Path

import IPython.display as ipyd
from IPython.display import HTML

# ── locate samples/ ────────────────────────────────────────────────────────
samples_dir = next(
    p / "samples"
    for p in [Path.cwd(), Path.cwd() / "src" / "classiflow" / "playground"]
    if (p / "samples").is_dir()
)
files = sorted(samples_dir.iterdir())
if not files:
    msg = f"No files in {samples_dir}. Drop a PDF/DOCX/image there first."
    raise FileNotFoundError(msg)

# ── process each file ───────────────────────────────────────────────────────
for file_path in files:
    file_bytes = file_path.read_bytes()
    repo = InMemoryAuditRepository()
    result = await FileReceptionAgent(
        audit=AuditService(repo),
        broadcaster=EventBroadcaster(),
        mime_detector=detect_mime,
    ).run(job_id="demo-real", filename=file_path.name, file_bytes=file_bytes)
    audit_records = await repo.list_for_job("demo-real")
    duration = audit_records[0].duration_ms if audit_records else None

    status_color = "#2e7d32" if result.passed else "#c62828"
    status_icon = "✅ PASSED" if result.passed else "❌ FAILED"
    sha_display = result.sha256[:16] + "…" if result.sha256 else "—"
    size_kb = f"{len(file_bytes) / 1024:.1f} KB"
    duration_str = f"{duration} ms" if duration is not None else "—"

    rows = [
        ("File", file_path.name),
        ("Size", size_kb),
        ("MIME", result.detected_mime or "—"),
        ("SHA-256 (prefix)", sha_display),
        ("Duration", duration_str),
        ("Rejection reason", result.rejection_reason or "—"),
    ]

    rows_html = "".join(
        f'<tr><td style="color:#555;padding:4px 12px 4px 0;white-space:nowrap">'
        f"{k}</td>"
        f'<td style="font-family:monospace;padding:4px 0">{v}</td></tr>'
        for k, v in rows
    )

    ipyd.display(
        HTML(f"""
        <div style="border:1px solid #ddd;border-radius:8px;padding:16px;
                    margin:8px 0;font-family:sans-serif;max-width:540px">
          <div style="font-size:1.1em;font-weight:bold;color:{status_color};
                      margin-bottom:10px">{status_icon}</div>
          <table style="border-collapse:collapse;width:100%">{rows_html}</table>
        </div>
        """)
    )

2026-06-21 21:46:01.463 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=demo-real agent=agent1_file_reception event=passed


File,convenio_2_2013.pdf
Size,118.3 KB
MIME,application/pdf
SHA-256 (prefix),5af08db117de1daf…
Duration,0 ms
Rejection reason,—


2026-06-21 21:46:01.477 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=demo-real agent=agent1_file_reception event=passed


File,DIA_A_Grupos_ACTUALIZADOS.xlsx
Size,10.0 KB
MIME,application/vnd.openxmlformats-officedocument.spreadsheetml.sheet
SHA-256 (prefix),f11ee60402bf021c…
Duration,0 ms
Rejection reason,—
